# Fraud Detection & Explainability

A production-grade ML pipeline that detects credit card fraud in real time and explains every decision using SHAP.

**Dataset:** [Sparkov Credit Card Fraud](https://www.kaggle.com/datasets/kartik2112/fraud-detection) — 1.85M synthetic transactions  
**Model:** LightGBM with business-driven threshold selection  
**Explainability:** SHAP TreeExplainer with waterfall plots  

## Pipeline
1. Load & validate data
2. Feature engineering (17 features)
3. Train LightGBM with class imbalance correction
4. Business-driven threshold selection (≥90% recall)
5. SHAP explainability — global + per-case

## 0. Setup

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from geopy.distance import geodesic
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, auc,
    precision_score, recall_score, f1_score, confusion_matrix
)

# ── paths ──────────────────────────────────────────────────────────
DATA_PATH  = "data"
MODELS_DIR = "models"
OUTPUTS    = "outputs"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

# ── skip flags — True if artifacts already exist ───────────────────
# this lets you re-run the notebook without recomputing slow steps
SKIP_DISTANCE = os.path.exists(f"{DATA_PATH}/train_with_distance.parquet")
SKIP_TRAINING = os.path.exists(f"{MODELS_DIR}/model.txt")
SKIP_SHAP     = os.path.exists(f"{MODELS_DIR}/shap_values.npy")

print(f"Skip distance computation: {SKIP_DISTANCE}")
print(f"Skip model training:       {SKIP_TRAINING}")
print(f"Skip SHAP computation:     {SKIP_SHAP}")

## 1. Load Data

In [ ]:
train_df = pd.read_csv(f"{DATA_PATH}/fraudTrain.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/fraudTest.csv")
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
train_df.head()

## 2. Exploratory Data Analysis

In [ ]:
fraud_rate = train_df['is_fraud'].mean()
print(f"Fraud rate: {fraud_rate:.4f} ({fraud_rate*100:.2f}%)")
print(train_df['is_fraud'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_df['hour_eda'] = pd.to_datetime(train_df['trans_date_trans_time']).dt.hour
hourly = train_df.groupby('hour_eda')['is_fraud'].mean()
axes[0].bar(hourly.index, hourly.values, color='steelblue')
axes[0].set_title('Fraud Rate by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate')
cat_fraud = train_df.groupby('category')['is_fraud'].mean().sort_values()
axes[1].barh(cat_fraud.index, cat_fraud.values, color='salmon')
axes[1].set_title('Fraud Rate by Category')
axes[2].hist(train_df[train_df['is_fraud']==0]['amt'].clip(upper=500), bins=50, alpha=0.6, label='Legit', color='steelblue')
axes[2].hist(train_df[train_df['is_fraud']==1]['amt'].clip(upper=500), bins=50, alpha=0.6, label='Fraud', color='salmon')
axes[2].set_title('Amount Distribution')
axes[2].legend()
plt.tight_layout()
plt.show()

## 3. Feature Engineering

17 features from raw transaction data:
- **Time:** hour, day_of_week, month, is_weekend, is_night
- **Age:** cardholder age at time of transaction
- **Distance:** geodesic distance between cardholder home and merchant
- **Amount:** log-transform + z-score within merchant category
- **City population:** log-transform
- **Categoricals:** merchant, category, gender, state, job (label encoded)

In [ ]:
train_df = pd.read_csv(f"{DATA_PATH}/fraudTrain.csv")
test_df  = pd.read_csv(f"{DATA_PATH}/fraudTest.csv")
CAT_COLS = ['merchant', 'category', 'gender', 'state', 'job']
DROP_IDENTITY = ['Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'trans_num', 'unix_time']
train_df = train_df.drop(columns=DROP_IDENTITY, errors='ignore')
test_df  = test_df.drop(columns=DROP_IDENTITY, errors='ignore')
for df in [train_df, test_df]:
    for col in CAT_COLS:
        df[f'{col}_original'] = df[col]
print("Identity columns dropped, originals saved")

In [ ]:
for df in [train_df, test_df]:
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour']        = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month']       = df['trans_date_trans_time'].dt.month
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['dob'] = pd.to_datetime(df['dob'])
    df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
print("Time and age features extracted")

In [ ]:
if SKIP_DISTANCE:
    print("Loading cached distances...")
    train_df['distance_km'] = pd.read_parquet(f"{DATA_PATH}/train_with_distance.parquet")['distance_km'].values
    test_df['distance_km']  = pd.read_parquet(f"{DATA_PATH}/test_with_distance.parquet")['distance_km'].values
else:
    print("Computing distances (takes ~10 mins)...")
    def compute_distance(row):
        try:
            return geodesic((row['lat'], row['long']), (row['merch_lat'], row['merch_long'])).km
        except:
            return np.nan
    train_df['distance_km'] = train_df.apply(compute_distance, axis=1)
    test_df['distance_km']  = test_df.apply(compute_distance, axis=1)
    train_df[['distance_km']].to_parquet(f"{DATA_PATH}/train_with_distance.parquet")
    test_df[['distance_km']].to_parquet(f"{DATA_PATH}/test_with_distance.parquet")
    print("Distance computed and cached")
print(f"Distance: mean={train_df['distance_km'].mean():.1f}km, max={train_df['distance_km'].max():.1f}km")

In [ ]:
for df in [train_df, test_df]:
    df['amt_log'] = np.log1p(df['amt'])
category_stats = train_df.groupby('category')['amt'].agg(['mean','std']).reset_index().rename(columns={'mean':'cat_amt_mean','std':'cat_amt_std'})
train_df = train_df.merge(category_stats, on='category', how='left')
test_df  = test_df.merge(category_stats, on='category', how='left')
for df in [train_df, test_df]:
    df['amt_zscore_by_category'] = (df['amt'] - df['cat_amt_mean']) / (df['cat_amt_std'] + 1e-8)
    df['city_pop_log'] = np.log1p(df['city_pop'])
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col]  = test_df[col].map(lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1)
    encoders[col] = le
DROP_AFTER = ['trans_date_trans_time','dob','lat','long','merch_lat','merch_long','zip','cat_amt_mean','cat_amt_std','city_pop','city']
train_df = train_df.drop(columns=DROP_AFTER, errors='ignore')
test_df  = test_df.drop(columns=DROP_AFTER, errors='ignore')
TARGET   = 'is_fraud'
exclude  = [TARGET, 'amt'] + [f'{col}_original' for col in CAT_COLS]
features = [col for col in train_df.columns if col not in exclude]
X_train = train_df[features]; y_train = train_df[TARGET]
X_test  = test_df[features];  y_test  = test_df[TARGET]
print(f"Features ({len(features)}): {features}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
joblib.dump(encoders,       f"{MODELS_DIR}/encoders.pkl")
joblib.dump(category_stats, f"{MODELS_DIR}/category_stats.pkl")
joblib.dump(features,       f"{MODELS_DIR}/train_columns.pkl")
print("Preprocessing artifacts saved")

## 4. Train LightGBM

Key decisions:
- `scale_pos_weight` corrects for ~0.5% fraud rate imbalance
- Threshold tuned on validation set for ≥90% recall
- Test set touched only once at final evaluation

In [ ]:
if SKIP_TRAINING:
    print("Loading existing model and threshold...")
    booster   = lgb.Booster(model_file=f"{MODELS_DIR}/model.txt")
    threshold = joblib.load(f"{MODELS_DIR}/threshold.pkl")
    features  = joblib.load(f"{MODELS_DIR}/train_columns.pkl")
    y_test_prob  = booster.predict(X_test)
    y_pred_tuned = (y_test_prob >= threshold).astype(int)
    print(f"Model loaded | threshold: {threshold:.3f}")
else:
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
    scale_pos_weight = len(y_tr[y_tr==0]) / len(y_tr[y_tr==1])
    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos_weight)
    model.fit(X_tr, y_tr)
    y_val_prob = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
    pr_auc     = auc(recalls, precisions)
    valid_mask = recalls[:-1] >= 0.90
    threshold  = thresholds[valid_mask][np.argmax(precisions[:-1][valid_mask])]
    print(f"PR-AUC: {pr_auc:.3f} | Threshold: {threshold:.3f}")
    model.booster_.save_model(f"{MODELS_DIR}/model.txt")
    joblib.dump(threshold, f"{MODELS_DIR}/threshold.pkl")
    y_test_prob  = model.predict_proba(X_test)[:, 1]
    y_pred_tuned = (y_test_prob >= threshold).astype(int)

## 5. Evaluation

In [ ]:
y_pred_default = (y_test_prob >= 0.5).astype(int)
comparison = pd.DataFrame({
    'Precision': [precision_score(y_test, y_pred_default), precision_score(y_test, y_pred_tuned)],
    'Recall':    [recall_score(y_test, y_pred_default),    recall_score(y_test, y_pred_tuned)],
    'F1':        [f1_score(y_test, y_pred_default),        f1_score(y_test, y_pred_tuned)],
}, index=[f'Default (0.50)', f'Tuned ({threshold:.2f})'])
print(comparison.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, title in zip(axes, [y_pred_default, y_pred_tuned], [f'Default (0.50)', f'Tuned ({threshold:.2f})']):
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    labels = np.array([[f"{v}\n({p:.1f}%)" for v, p in zip(rv, rp)] for rv, rp in zip(cm, cm_pct)])
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. SHAP Explainability

In [ ]:
booster  = lgb.Booster(model_file=f"{MODELS_DIR}/model.txt")
explainer = shap.TreeExplainer(booster)
if SKIP_SHAP:
    print("Loading cached SHAP values...")
    shap_values = np.load(f"{MODELS_DIR}/shap_values.npy")
else:
    print("Computing SHAP values (takes a few minutes)...")
    shap_values = explainer.shap_values(X_test)
    np.save(f"{MODELS_DIR}/shap_values.npy", shap_values)
    print("Cached")
print(f"Shape: {shap_values.shape}")

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title("Global Feature Importance — SHAP Beeswarm", fontsize=13, pad=12)
plt.savefig(f"{OUTPUTS}/shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.title("Global Feature Importance — Mean |SHAP|", fontsize=13, pad=12)
plt.savefig(f"{OUTPUTS}/shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-Case Waterfall Explanations

In [ ]:
X_test_copy = X_test.copy()
X_test_copy['predicted_prob']  = y_test_prob
X_test_copy['predicted_class'] = y_pred_tuned
X_test_copy['actual']          = y_test.values
for col in ['amt','category_original','merchant_original','job_original','state_original','gender_original','distance_km']:
    if col in test_df.columns:
        X_test_copy[col] = test_df.loc[X_test_copy.index, col].values
correct_fraud = X_test_copy[(X_test_copy['actual']==1) & (X_test_copy['predicted_class']==1)]
cases = correct_fraud.sample(4, random_state=20)
print(f"True positives: {len(correct_fraud)} | Selected: {len(cases)}")

In [ ]:
explanation = explainer(X_test, check_additivity=False)
def sigmoid(x): return 1 / (1 + np.exp(-x))
raw_base = explanation.base_values
prob_base = sigmoid(raw_base)
prob_output = sigmoid(raw_base + explanation.values.sum(axis=1))
scale = np.where(explanation.values.sum(axis=1)!=0, (prob_output-prob_base)/explanation.values.sum(axis=1), 1.0)
prob_explanation = shap.Explanation(
    values=explanation.values * scale[:, None],
    base_values=prob_base, data=explanation.data,
    feature_names=explanation.feature_names
)
print("SHAP Explanation object ready")

In [ ]:
DAYS = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
for i, (idx, row) in enumerate(cases.iterrows(), start=1):
    pos      = X_test.index.get_loc(idx)
    merchant = str(row.get('merchant_original','unknown')).replace('fraud_','')
    category = str(row.get('category_original','unknown')).replace('_',' ')
    plt.figure()
    shap.plots.waterfall(prob_explanation[pos], max_display=12, show=False)
    plt.title(f"Case {i}  |  {row['predicted_prob']:.0%} fraud probability\n${row['amt']:.2f} at {merchant} ({category})", fontsize=10, pad=12)
    path = f"{OUTPUTS}/waterfall_case_{i}.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    day = DAYS[int(row['day_of_week'])]
    print(f"Case {i}: {row['predicted_prob']:.0%} | ${row['amt']:.2f} at {merchant} | {day} {int(row['hour'])}:00 | {row['distance_km']:.1f}km from home")

In [ ]:
cases.drop(columns=['shap_top_features'], errors='ignore').to_csv(f"{OUTPUTS}/explained_cases.csv")
print("All outputs saved!")
print(f"Threshold: {threshold:.4f}")
print(f"Precision: {precision_score(y_test, y_pred_tuned):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_tuned):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_tuned):.3f}")